In [ ]:
import sys; print(sys.executable)

In [ ]:
import os
os.chdir("../")
print(os.getcwd())

### 03
- msptのavgとminに相関があるのは理解できるが、逆にそれらとmaxに相関がないのがわからない
- 強い相関のものでも0.95は超えていない
- msptとheapに弱い負の相関が見られる


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from preprocessing.feature_engineer import add_gc_rate

df = pd.read_parquet("data/processed/server_metrics_20260503.parquet")

df = add_gc_rate(df)    # rate化
df = df.set_index("timestamp")
df = df.iloc[:-1]   # 監視中止の最終行除去
df = df.drop(columns=[
    "gc_count_total", "gc_time_ms_total",   # 累積カラム除去
    "tps", "heap_max_mb", "thread_count"   # 定数カラム除去
])
df = df.dropna()

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(df), index=df.index, columns=df.columns)
X.describe().round(2)


In [ ]:
# 相関ヒートマップ
import numpy as np
import matplotlib.pyplot as plt

corr = df.corr()    # Pearson

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)  # 0中心の発散カラーマップ

ax.set_xticks(range(len(corr.columns)), labels=corr.columns, rotation=90)
ax.set_yticks(range(len(corr.columns)), labels=corr.columns)

for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)

fig.colorbar(im, ax=ax)
ax.set_title("Feature correlation (Pearson)")
plt.tight_layout()
plt.show()


### 04
- msptのavgとminに0.95+が見られる
- msptとonline_playersに負の相関が見られる

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from preprocessing.feature_engineer import add_gc_rate

df = pd.read_parquet("data/processed/server_metrics_20260504.parquet")

df = add_gc_rate(df)    # rate化
df = df.set_index("timestamp")
df = df.iloc[:-1]   # 監視中止の最終行除去
df = df.drop(columns=[
    "gc_count_total", "gc_time_ms_total",   # 累積カラム除去
    "tps", "heap_max_mb", "thread_count"   # 定数カラム除去
])
df = df.dropna()

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(df), index=df.index, columns=df.columns)
X.describe().round(2)


In [ ]:
# 相関ヒートマップ
import numpy as np
import matplotlib.pyplot as plt

corr = df.corr()    # Pearson

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)  # 0中心の発散カラーマップ

ax.set_xticks(range(len(corr.columns)), labels=corr.columns, rotation=90)
ax.set_yticks(range(len(corr.columns)), labels=corr.columns)

for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)

fig.colorbar(im, ax=ax)
ax.set_title("Feature correlation (Pearson)")
plt.tight_layout()
plt.show()


### 03 + 04
- 結合の結果04で見られた特徴はサンプル数による偏りであると考えられる
- `mspt` の `max` は `min` / `avg` とは独立している
    - スパイク型異常の信号は `max` に集約される
- 相関は支配的な程ではない


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from preprocessing.feature_engineer import add_gc_rate

df1 = pd.read_parquet("data/processed/server_metrics_20260503.parquet")
df2 = pd.read_parquet("data/processed/server_metrics_20260504.parquet")
df = pd.concat([df1, df2])

df = add_gc_rate(df)    # rate化
df = df.set_index("timestamp")
df = df.iloc[:-1]   # 監視中止の最終行除去
df = df.drop(columns=[
    "gc_count_total", "gc_time_ms_total",   # 累積カラム除去
    "tps", "heap_max_mb", "thread_count"   # 定数カラム除去
])
df = df.dropna()

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(df), index=df.index, columns=df.columns)
X.describe().round(2)



In [ ]:
import matplotlib.pyplot as plt

# 相関ヒートマップ
corr = df.corr()    # Pearson

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)  # 0中心の発散カラーマップ

ax.set_xticks(range(len(corr.columns)), labels=corr.columns, rotation=90)
ax.set_yticks(range(len(corr.columns)), labels=corr.columns)

for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)

fig.colorbar(im, ax=ax)
ax.set_title("Feature correlation (Pearson)")
plt.tight_layout()
plt.show()
